# Lesson 28 Lab — Capacity, Cost, and Autoscaling

**Puzzle:** How many replicas are required when average traffic is comfortable but the peak SLO is not?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Capacity planning converts a measured service curve into replicas and headroom. Average tokens per second hides burstiness, prompt/decode mix, failure reserve, and the latency cliff near saturation.


## 0. Predict before running

1. Compute replicas at 50%, 70%, and 85% utilization targets.
2. Add N+1 reserve.
3. Choose a pre-saturation scaling signal.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab anchors a queue/capacity model to a small native throughput measurement on the RTX 5090, then calculates replicas for three demand scenarios with utilization and N+1 reserve.

- Saturation throughput is not an SLO-safe operating point.
- Headroom covers burst, variance, and failure—not just growth.
- Scale-up delay must be compared with traffic forecast horizon.


## 2. Derive the mechanism

Usable replica capacity is measured token throughput multiplied by a safe utilization target, not the saturation maximum. Required replicas are the ceiling of demand divided by usable capacity, then adjusted for availability and heterogeneous prompt cost. Autoscaling signals need lead time because new replicas load gigabytes of weights.

### Mechanism at a glance

```mermaid
flowchart LR
  M["measured service curve"] --> U["safe utilization ceiling"]
  D["forecast peak demand"] --> R["ceil demand / usable capacity"]
  U --> R
  R --> N["N+1 / zone reserve"]
  L["model load lead time"] --> A["autoscaling trigger"]
  N --> A
```

### Walk it step by step

1. **Measure a service curve.** Find throughput that still satisfies the latency SLO.
2. **Choose operating headroom.** Reserve capacity for variance and recovery.
3. **Calculate failure-aware replicas.** Add the declared availability reserve.
4. **Trigger before saturation.** Account for image/model load and readiness delay.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 28
LESSON_TITLE = 'Capacity, Cost, and Autoscaling'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260840
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | replicas from average demand and peak throughput |
| Candidate | SLO-safe utilization plus peak demand, N+1, and load delay |
| Held constant | measured model/GPU rate, demand scenarios, utilization target, and availability policy |
| Measurements | native rate, usable rate, base replicas, reserved replicas, utilization, and scale trigger |
| Evidence | `capacity-model` |

**Experiment:** Measure one native token rate and feed it into an explicit replica/autoscaling table.


## 5. Inspect the experiment code

The measurement and planning arithmetic stay in separate fields. A scenario never changes the measured RTX 5090 throughput value.

Do not execute until the code matches the frozen table.


In [2]:
llm=LLM(**base_engine_args(max_model_len=1024,max_num_seqs=8)); params=SamplingParams(temperature=0.0,max_tokens=24,seed=SEED)
prompts=[f"Capacity probe {i}: name one serving metric." for i in range(8)]
llm.generate(["warmup"],SamplingParams(temperature=0.0,max_tokens=2),use_tqdm=False)
tick=time.perf_counter(); outputs=llm.generate(prompts,params,use_tqdm=False); elapsed=time.perf_counter()-tick
tokens=sum(len(x.outputs[0].token_ids) for x in outputs); measured=tokens/elapsed; safe=.65; usable=measured*safe
demands={"low":measured*.3,"medium":measured*1.4,"peak":measured*3.2}; scenarios={}
for name,demand in demands.items():
    base=max(1,math.ceil(demand/usable)); scenarios[name]={"demand_output_tokens_s":demand,
        "base_replicas":base,"with_reserve":base+1,"projected_utilization":demand/(base*measured)}
metrics={"measured_output_tokens_s":measured,"measurement_elapsed_s":elapsed,"measurement_output_tokens":tokens,
 "safe_utilization":safe,"usable_output_tokens_s":usable,"scenarios":scenarios,"scale_up_lead_s":180.0}
analysis=(f"The native closed batch measured {measured:.1f} output tokens/s. At {safe:.0%} safe "
          f"utilization, medium/peak need {scenarios['medium']['with_reserve']}/"
          f"{scenarios['peak']['with_reserve']} replicas including N+1. Online service curves remain required.")


INFO 08-13 00:25:26 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260840, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 8, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:25:26 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:25:26 [model.py:1883] Using max model len 1024


INFO 08-13 00:25:26 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:25:26 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=654081) 2026-08-13 00:25:37,617 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 0 configs from <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json
(EngineCore pid=654081) 2026-08-13 00:25:37,617 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=654081) 2026-08-13 00:25:37,675 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=654081) 2026-08-13 00:25:37,681 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=654081) INFO 08-13 00:25:43 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=654081) INFO 08-13 00:25:43 [core.py:355] init engine (profile, create kv cache, warmup model) took 7.60 s


(EngineCore pid=654081) WARNING 08-13 00:25:43 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=654081) WARNING 08-13 00:25:43 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=654081) INFO 08-13 00:25:43 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=654081) INFO 08-13 00:25:43 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=654081) INFO 08-13 00:25:43 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:25:44 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Measured output tok/s | 820.7/s |
| Safe utilization | 65.00% |
| Usable tok/s | 533.4/s |
| Medium replicas | 4 |
| Peak replicas | 6 |
| Scale-up lead | 180.000000 |


## 7. Explain the result

The native closed batch measured 820.7 output tokens/s. At 65% safe utilization, medium/peak need 4/6 replicas including N+1. Online service curves remain required.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. Measured environment facts feed explicit planning arithmetic. Assumed topology, demand, bandwidth, and reserve fields remain assumptions until a native deployment test.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 28, "title": 'Capacity, Cost, and Autoscaling', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Replica arithmetic must be anchored to an SLO-safe service curve; the current native rate is a laboratory input, not production capacity.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 28,
  "title": "Capacity, Cost, and Autoscaling",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260840
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "measured_output_tokens_s": 820.6668715815052,
    "measurement_elapsed_s": 0.2339560748077929,
    "measurement_output_tokens": 192,
    "safe_utilization": 0.65,
    "usable_output_tokens_s": 533.4334665279785,
    "scenarios": {
      "low": {
        "demand_output_tokens_s": 246.20006147445156,
        "base_replicas": 1,
        "with_reserve": 2,
        "projected_utilization": 0.3
      },
      "medium": {
        "demand_output_tokens_s": 1148.9336202141073,
        "base_replicas": 3,
        "with_reserve": 4,
        "projected_utilization": 0.4666666666666667
      },
      "peak": {
        "d

## 9. Make the bounded decision

> Replica arithmetic must be anchored to an SLO-safe service curve; the current native rate is a laboratory input, not production capacity.

**Acceptance/rollback:** Provision only after peak-trace replay confirms the chosen replica count meets TTFT/ITL/error gates with one replica unavailable.

**Failure analysis:** One offline rate omits online queueing and prompt work. Cost varies by provider, reservation, power, and utilization; this lesson does not quote a currency price.


## 10. Extend the evidence

Build an online service curve, inject a replica failure at peak, measure startup time, and validate predictive or queue-based autoscaling before production.

The full boundary and references are in [`README.md`](README.md).
